# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [10]:
!pip install -q duckdb scikit-learn
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    average_precision_score
)

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/**/*.parquet"
)

print("Setup complete.")

Setup complete.


In [11]:
march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet('{march_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

march_features = con.sql(march_query).df()

march_features.head()
april_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS april_impressions

FROM read_parquet('{april_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcome = con.sql(april_query).df()

april_outcome.head()
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)
model_df["impression_change_pct"] = (
    (
        model_df["april_impressions"]
        - model_df["gsc_impressions"]
    )
    / model_df["gsc_impressions"]
) * 100

model_df["decline_label"] = (
    model_df["impression_change_pct"] < -20
).astype(int)
print("Rows:", len(model_df))

print("\nLabel counts:")
print(model_df["decline_label"].value_counts())

print("\nLabel rate:")
print(
    model_df["decline_label"]
    .mean()
    .round(4)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549

Label counts:
decline_label
0    82738
1    75811
Name: count, dtype: int64

Label rate:
0.4782


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Method choice and why

I chose Logistic Regression because my target is binary: whether a content item experiences a greater-than-20% decline in Google Search impressions after the March decision window.

Logistic Regression is a suitable first model because it is simple, interpretable, and produces a probability that can be used to rank content for review.

The goal is not to maximize model complexity. The model must provide useful ranking information and improve on the Week-4 rule baseline on the same held-out data and metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped split by `client_hash_id` so that a client does not appear in both the training and test sets.

This reduces the risk that the model learns client-specific patterns that would make the evaluation look better than it should.

The split is fixed with a random seed so the experiment is reproducible. The Week-4 baseline will be evaluated on exactly the same held-out test rows.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]

required_columns = (
    feature_columns
    + ["decline_label", "client_hash_id", "content_hash_id"]
)

model_df = model_df.dropna(
    subset=required_columns
).reset_index(drop=True)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["decline_label"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Shared clients:",
    len(
        set(train_df["client_hash_id"])
        &
        set(test_df["client_hash_id"])
    )
)

Train rows: 93206
Test rows: 20647
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_df[feature_columns]
y_train = train_df["decline_label"]

X_test = test_df[feature_columns]
y_test = test_df["decline_label"]

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(
    X_train,
    y_train
)

test_probability = model.predict_proba(
    X_test
)[:, 1]

test_prediction = (
    test_probability >= 0.5
).astype(int)

In [15]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1][:k]

    return y_true[order].mean()

model_precision_50 = precision_at_k(
    y_test.to_numpy(),
    test_probability,
    k=50
)

print(
    "Model Precision@50:",
    round(model_precision_50, 4)
)

Model Precision@50: 0.66


In [18]:
print(baseline_test.columns.tolist())

# Make fresh copies
baseline_train = train_df.copy()
baseline_test = test_df.copy()

# -----------------------------
# 1. Calculate CTR
# -----------------------------

baseline_train["gsc_ctr"] = np.where(
    baseline_train["gsc_impressions"] > 0,
    baseline_train["gsc_clicks"] / baseline_train["gsc_impressions"],
    0
)

baseline_test["gsc_ctr"] = np.where(
    baseline_test["gsc_impressions"] > 0,
    baseline_test["gsc_clicks"] / baseline_test["gsc_impressions"],
    0
)


# -----------------------------
# 2. Create percentile function
# -----------------------------

def percentile_score(train_values, test_values):
    train_values = pd.Series(train_values).dropna().to_numpy()
    test_values = pd.Series(test_values).to_numpy()

    train_values = np.sort(train_values)

    scores = np.searchsorted(
        train_values,
        test_values,
        side="right"
    ) / len(train_values)

    return scores


# -----------------------------
# 3. Impression score
# -----------------------------

baseline_test["impression_score"] = percentile_score(
    baseline_train["gsc_impressions"],
    baseline_test["gsc_impressions"]
)


# -----------------------------
# 4. Position opportunity
# -----------------------------

baseline_test["position_opportunity"] = percentile_score(
    baseline_train["gsc_avg_position"],
    baseline_test["gsc_avg_position"]
)


# -----------------------------
# 5. CTR opportunity
# -----------------------------

baseline_test["ctr_opportunity"] = (
    1 -
    percentile_score(
        baseline_train["gsc_ctr"],
        baseline_test["gsc_ctr"]
    )
)


# -----------------------------
# 6. W04 baseline score
# -----------------------------

baseline_test["baseline_score"] = (
    0.4 * baseline_test["impression_score"]
    +
    0.3 * baseline_test["position_opportunity"]
    +
    0.3 * baseline_test["ctr_opportunity"]
)


print("Baseline score created successfully!")

baseline_test[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "gsc_ctr",
        "impression_score",
        "position_opportunity",
        "ctr_opportunity",
        "baseline_score"
    ]
].head()

['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'sessions_organic', 'ga4_engaged_sessions', 'april_impressions', 'impression_change_pct', 'decline_label', 'gsc_ctr']
Baseline score created successfully!


,gsc_impressions,gsc_avg_position,gsc_ctr,impression_score,position_opportunity,ctr_opportunity,baseline_score
2966,37.0,7.759091,0.000000,0.244544,0.469283,0.444478,0.371946
2967,1655.0,5.122100,0.002417,0.768599,0.269854,0.256582,0.465370
2968,82.0,9.530220,0.000000,0.327769,0.551681,0.444478,0.429955
2969,43.0,12.074815,0.000000,0.259340,0.622052,0.444478,0.423695
2970,568.0,14.167597,0.007042,0.604800,0.665687,0.096432,0.470556


In [19]:
baseline_precision_50 = precision_at_k(
    baseline_test["decline_label"].to_numpy(),
    baseline_test["baseline_score"].to_numpy(),
    k=50
)

print(
    "W04 Baseline Precision@50:",
    round(baseline_precision_50, 4)
)
test_probability

model_precision_50 = precision_at_k(
    y_test.to_numpy(),
    test_probability,
    k=50
)

print(
    "Logistic Regression Precision@50:",
    round(model_precision_50, 4)
)

W04 Baseline Precision@50: 0.42
Logistic Regression Precision@50: 0.66


In [20]:
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": [
        "W04 baseline",
        "Logistic Regression",
        "Test-set base rate"
    ],
    "precision_at_50": [
        baseline_precision_50,
        model_precision_50,
        base_rate
    ]
})

comparison

,method,precision_at_50
0,W04 baseline,0.420000
1,Logistic Regression,0.660000
2,Test-set base rate,0.525742


### Model vs baseline

On the same held-out test set, the Week-4 baseline achieved a Precision@50 of 0.420, while Logistic Regression achieved a Precision@50 of 0.660.

This means that 21 of the top 50 items ranked by the baseline were positive cases, compared with 33 of the top 50 items ranked by Logistic Regression.

The test-set base rate was 0.526, so the Logistic Regression model also performed above the overall rate of positive cases.

The model therefore outperformed the Week-4 baseline by 0.24 percentage points on Precision@50 for this held-out split.

This result suggests that the learned model provides a stronger ranking signal than the simple rule in this experiment. However, it does not prove that the model will generalize to every client or future time period, and it does not establish that a content refresh will cause impressions to recover.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import precision_score, recall_score, average_precision_score

precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

average_precision = average_precision_score(
    y_test,
    test_probability
)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("Average Precision:", round(average_precision, 4))

Precision: 0.6309
Recall: 0.0271
Average Precision: 0.5903


In [22]:
errors = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "ga4_engaged_sessions",
        "decline_label"
    ]
].copy()

errors["predicted_probability"] = test_probability

errors["predicted_label"] = test_prediction

errors["error"] = (
    errors["predicted_label"]
    != errors["decline_label"]
)

wrong_cases = errors[
    errors["error"]
].copy()

print("Total test rows:", len(errors))
print("Wrong predictions:", len(wrong_cases))

wrong_cases.head(10)

Total test rows: 20647
Wrong predictions: 10733


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions,decline_label,predicted_probability,predicted_label,error
2974,client_def0955f7a377868,content_f7fbef487d4e7f02,104.0,0.0,8.399513,0.0,0.0,1,0.471682,0,True
2976,client_def0955f7a377868,content_fd7d311b8f7b4170,23.0,0.0,7.486111,0.0,0.0,1,0.472010,0,True
2981,client_def0955f7a377868,content_e29b9565dafefc47,346.0,0.0,8.045437,0.0,0.0,1,0.473365,0,True
2984,client_fef1a8f436438636,content_184ea41438ed3eb6,13876.0,16.0,2.968057,15.0,1.0,0,0.506406,1,True
2985,client_fef1a8f436438636,content_27bb6eb0be0326f8,868.0,5.0,6.443521,6.0,0.0,1,0.462251,0,True
2987,client_fef1a8f436438636,content_af46af36db1a7123,1122.0,6.0,5.046418,7.0,0.0,1,0.461356,0,True
2988,client_fef1a8f436438636,content_af065f0adab1929e,759.0,1.0,6.368998,2.0,0.0,1,0.475883,0,True
2989,client_fef1a8f436438636,content_2ee618c04da55353,3232.0,13.0,3.941933,25.0,0.0,1,0.474396,0,True
2991,client_fef1a8f436438636,content_25e74b5e3fb2f4dc,9454.0,70.0,4.041613,111.0,3.0,1,0.405895,0,True
2992,client_fef1a8f436438636,content_c2ff8b658ca4e989,5376.0,32.0,4.154883,43.0,2.0,1,0.438129,0,True


## 4. Errors and interpretation

The Logistic Regression model achieved a precision of 0.6309 and a recall of 0.0271 at the default 0.5 classification threshold. The low recall should not be interpreted as a failure by itself because this lane is primarily a prioritization problem. The main decision is which content items should be reviewed first, so ranking metrics such as Precision@50 are more relevant than classifying every row correctly.

The model achieved an Average Precision of 0.5903 compared with a test-set positive rate of 0.5257. This indicates that the model has some ability to rank higher-risk items ahead of lower-risk items, although the improvement across the full ranking is moderate.

The most useful result is the top-of-ranking comparison. Logistic Regression achieved Precision@50 of 0.660, compared with 0.420 for the Week-4 baseline. Therefore, 33 of the model's top 50 items were positive cases, compared with 21 of the baseline's top 50.

The model still makes substantial errors. For example, one false negative had 104 impressions, zero clicks, and an average position of about 8.4, but its predicted decline probability was only 0.472. This shows that the available features do not capture every future decline.

A false positive example had 13,876 impressions and an average position of about 2.97, but the model assigned a decline probability of about 0.506 even though the observed label was 0. This demonstrates why the prediction should be treated as decision support rather than an automatic refresh decision.

These errors suggest that the model captures useful patterns but does not provide certainty about future performance.

In [23]:
coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

,feature,coefficient,abs_coefficient
1,gsc_clicks,-0.765112,0.765112
3,sessions_organic,0.240358,0.240358
0,gsc_impressions,0.145894,0.145894
4,ga4_engaged_sessions,0.064926,0.064926
2,gsc_avg_position,-0.051644,0.051644


### Feature interpretation

The Logistic Regression coefficients are based on standardized features, so their magnitudes can be compared to understand which signals contribute most strongly to the model.

`gsc_clicks` has the largest absolute coefficient (-0.7651). Its negative coefficient means that higher click values are associated with lower predicted decline risk, holding the other features constant.

`sessions_organic` has the second-largest coefficient (+0.2404), meaning higher organic sessions are associated with higher predicted decline risk in this model. This is not interpreted causally; it may reflect differences in content or client mix, or relationships between the available traffic measures.

`gsc_impressions` has a positive coefficient (+0.1459), indicating that higher impressions are associated with higher predicted decline risk.

`ga4_engaged_sessions` has a smaller positive coefficient (+0.0649), while `gsc_avg_position` has the smallest absolute coefficient (-0.0516).

The coefficients describe associations learned from the training data and should not be interpreted as causal effects. In particular, the unexpected positive association for organic sessions should be investigated further rather than treated as evidence that organic sessions cause future decline.

### What surprised me

The strongest feature was `gsc_clicks`, with a coefficient of -0.7651. I expected impressions and search position to be important because they were central to the Week-4 rule, but the model relied more heavily on clicks.

The positive coefficient for `sessions_organic` was also unexpected. I do not have enough evidence to explain this as a causal relationship, so I treat it as an association that requires further investigation.

This is a useful reminder that a model can combine signals differently from a manually designed rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.